# Capstone — FCFS vs LTR vs PARS+ProD-M+Priority

**Primary scale: 1000 prompts** (`configs/live_run.yaml`), saved every **50** prompts to Drive.

1. Runtime → GPU (A100 / L4 preferred)
2. Accept: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
3. Paste HF token below
4. Run cells in order (`--resume` safe after disconnects)

In [ ]:
import os
from google.colab import drive

os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # <-- paste token
drive.mount("/content/drive")

BACKUP = "/content/drive/MyDrive/capstone_results"
os.makedirs(BACKUP, exist_ok=True)

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import shutil, os, subprocess, sys

ROOT = "/content/pairwise-ltr-scheduler"
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)

subprocess.check_call([
    "git", "clone",
    "https://github.com/anmolsaluja/pairwise-ltr-scheduler.git", ROOT,
])
os.chdir(ROOT)

import torch
print("cuda before install:", torch.cuda.is_available(), torch.__version__)
if not torch.cuda.is_available():
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch",
        "--index-url", "https://download.pytorch.org/whl/cu121",
    ])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

BACKUP = "/content/drive/MyDrive/capstone_results"
os.makedirs("data/processed", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
for name in ("prod_labels.json", "prod_hidden.pt"):
    src = f"{BACKUP}/{name}"
    if os.path.exists(src):
        shutil.copy2(src, f"data/processed/{name}")
        print("restored", name)
for name in ("ltr_pointwise.pt", "pairwise_ranker.pt"):
    src = f"{BACKUP}/checkpoints/{name}"
    if os.path.exists(src):
        shutil.copy2(src, f"checkpoints/{name}")
        print("restored", name)

!python scripts/check_setup.py

In [ ]:
# Label generation: 1000 prompts, save every 50
!python scripts/generate_labels.py --config configs/live_run.yaml \
  --limit 1000 \
  --chunk-size 50 \
  --num-samples 3 \
  --resume \
  --device cuda \
  --backup-dir /content/drive/MyDrive/capstone_results

In [ ]:
import json
path = "data/processed/prod_labels.json"
with open(path) as f:
    n = len(json.load(f)["records"])
print(f"Labeled prompts: {n}/1000")
if n < 1000:
    print("Re-run the previous cell until you see 1000/1000")
else:
    print("Ready for train + evaluate (next cell)")

In [ ]:
# Train LTR + PARS + evaluate (after 1000 labels)
!python scripts/train_prod_m.py --config configs/live_run.yaml --target single --output checkpoints/ltr_pointwise.pt --device cuda
!python scripts/train_ranker.py --config configs/live_run.yaml --train-samples 1000 --device cuda
!python scripts/evaluate.py --config configs/live_run.yaml --limit 1000 --device cuda

!mkdir -p /content/drive/MyDrive/capstone_results/checkpoints
!cp -r checkpoints data/processed /content/drive/MyDrive/capstone_results/
print("checkpoints + labels saved to Drive")

## Results section (report figures)

Main-paper style graphs for **1000 prompts**: length distributions, latency vs rate, FCFS/LTR/OURS bars.

In [ ]:
import os, glob, subprocess, sys
from IPython.display import display, Markdown, Image

FIG = "/content/drive/MyDrive/capstone_results/figures"
os.makedirs(FIG, exist_ok=True)
LIMIT = 1000

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
rc = subprocess.call([
    sys.executable, "-u", "scripts/plot_results.py",
    "--config", "configs/live_run.yaml",
    "--limit", str(LIMIT),
    "--device", "cuda",
    "--out-dir", FIG,
])
assert rc == 0, "plot_results failed — scroll up for the error"

md_path = f"{FIG}/results_section.md"
if os.path.exists(md_path):
    display(Markdown(open(md_path).read()))

print("\nFigures saved to:", FIG)
for path in sorted(glob.glob(f"{FIG}/fig_*.png")):
    print(path)
    display(Image(filename=path))

### If Colab disconnects during labeling

1. Re-run token + clone cells  
2. Re-run the **label generation** cell (`--resume` + Drive backup)  
3. Continue until **1000/1000**  
4. Then train + evaluate + Results graphs